In [1]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import logging

logging.basicConfig(level=logging.INFO)
logger=logging.getLogger(__name__)

spark=SparkSession.builder.appName("starting_etl_ingestion").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
import os
from datetime import datetime


mysql_config={
        
        "host": os.getenv("host", "host.docker.internal"),
        "port": os.getenv("port", "3306"),
        "database": os.getenv("database", "ecommerce"),
        "username": os.getenv("username","root"),
        "password": os.getenv("password","Chiku@123"),
        "driver": os.getenv("driver", "com.mysql.cj.jdbc.Driver")
    }
url=f"jdbc:mysql://{mysql_config['host']}:{mysql_config['port']}/{mysql_config['database']}"


bucket="e-commerce-sales-etl"
prefix="ingestion_data"
base_path=f"s3://{bucket}/{prefix}"
bronze_prefix="bronze_data"
bronze_path=f"s3://{bucket}/{bronze_prefix}"
timestamp=datetime.now().strftime("%Y%m%d")


table_name=[{
            "source_type":"my_sql",
            "source_table":"customers",
            "target_table":"customer_data"
            },
            {
            "source_type":"my_sql",
            "source_table":"products",
            "target_table":"product_data"
            },
            {"source_type":"s3",
            "source_table":"inventory_data",
            "target_table":"inventory_data"
            },
            {"source_type":"s3",
            "source_table":"order_item_data",
            "target_table":"order_item_data"
                },
            {"source_type":"s3",
            "source_table":"orders_data",
            "target_table":"orderss_data"
            }
             ]

for table in table_name:
        if table["source_type"]=="my_sql":

            target_table=table["target_table"]
            s3_bronze_path=f"{bronze_path}/{target_table}/{timestamp}"
            print(s3_bronze_path)

"""for table in table_name:
        if table["source_type"]=="my_sql":
            print(f"{table['source_table']}")
        elif table["source_type"]=="s3":
            s3_path=f"{base_path}/{table['source_table']}"
            df=spark.read.option("multiline","True").json(s3_path)
            df.show()"""

s3://e-commerce-sales-etl/bronze_data/customer_data/20260822
s3://e-commerce-sales-etl/bronze_data/product_data/20260822


'for table in table_name:\n        if table["source_type"]=="my_sql":\n            print(f"{table[\'source_table\']}")\n        elif table["source_type"]=="s3":\n            s3_path=f"{base_path}/{table[\'source_table\']}"\n            df=spark.read.option("multiline","True").json(s3_path)\n            df.show()'

In [ ]:
""" 
diff between resources and client for calling aws services (when to use and why )
Simple CRUD/control-table operations → resource
Low-level control, specific DynamoDB APIs, or fine-grained API handling → client

So if you want to learn the AWS internals and you're comfortable with the extra syntax, continue with client for this project. Just remember that it's not inherently more performant or more "industry standard" than resource."""
import boto3
#Extracting the timestamp from dynmodb
dynomodb=boto3.resource("dynamodb")
table= dynomodb.Table("ecommerce_pipeline_control")
                            
                            
response=table.get_item(Key={"pipeline_table":"orders"})

print(response)
#Extrcating the timestamp from s3

last_successful_watermark=response["Item"]["last_successful_watermark"]
print(last_successful_watermark)

from datetime import datetime
last_successful_watermark=datetime.now().strftime("%Y%m%d_%H%M%S")


{'Item': {'last_run_status': 'intial', 'last_run_timestamp': '1970-01-01 00:00:00', 'Source_name': 'orders', 'pipeline_table': 'orders', 'record_processed': Decimal('0'), 'Source_type': 's3', 'last_successful_watermark': '1970-01-01 00:00:00'}, 'ResponseMetadata': {'RequestId': '48EASHG1UANFJSUE6E164SMIHNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sat, 22 Aug 2026 04:49:59 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '263', 'connection': 'keep-alive', 'x-amzn-requestid': '48EASHG1UANFJSUE6E164SMIHNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '28339119'}, 'RetryAttempts': 0}}
1970-01-01 00:00:00


In [ ]:
import boto3
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, DataFrame, max
import logging


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

spark = SparkSession.builder.appName("starting_etl_ingestion").getOrCreate()

logging.info("starting etl ingestion")


def ingest_data_from_s3(table: str,s3_path:str, s3_target_path: str) -> DataFrame:
    df = spark.read.option("multiline", "true").json(s3_path)

    dynamodb = boto3.resource("dynamodb")
    d_table = dynamodb.Table("ecommerce_pipeline_control")
    response = d_table.get_item(Key={"pipeline_table": "orders"})
    print(response)

    last_successful_watermark = response.get("Item", {}).get("last_successful_watermark")
    print(last_successful_watermark)

    #Incremental startegy:
    incremental_data_df=df.filter(col("last_updated_timestamp")>(last_successful_watermark))
    new_data_count = incremental_data_df.count()
    if new_data_count>0:
    #Find maximaum processed_timestamp
        max_load_timestamp=incremental_data_df.agg(max(col("last_updated_timestamp")).alias("max_load_timestamp")).collect()[0]["max_load_timestamp"]

        #Loading the  data to target
        incremental_data_df.write.mode("overwrite").parquet(s3_target_path)

        #Audit information
        from datetime import datetime
        last_run_timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        #Updating dynomodb paramters
        record_processed=new_data_count
        update_watermark=max_load_timestamp
        d_table.update_item(Key={"pipeline_table":"orders"},
                        UpdateExpression="""
                                        SET last_successful_watermark = :wm,
                                            last_run_status = :status,
                                            last_run_timestamp = :run_time,
                                            records_processed = :count
                                        """,
                        ExpressionAttributeValues={
                                ":wm": update_watermark,
                                ":status": "SUCCESS",
                                ":run_time": last_run_timestamp,
                                ":count": record_processed
                        }
                )
    else:
        logger.info("no incremental data found")



                                                      






def ingest_data_from_mysql_database(table:str,s3_target_path:str) ->DataFrame:
    df=spark.read.jdbc(
    url=url,
    table=table,
    properties={
    "user":mysql_config["username"],
    "password":mysql_config["password"],
    "driver":mysql_config["driver"]
    }
    
    )

    #Incremental startegy:
    #incremental_data_df=df.filter(col("last_processed_timestamp")>(col("last_watermark_timestamp")))
    
    #Extracting the timestamp from dynmodb
    #dynomodb=boto3.client("dynamodb")
    #last_watermark_timestamp= dynomodb.
    #Extrcating the timestamp from s3

    #Loading the incremental data to source
    #df.write.mode("overwrite").parquet(s3_target_path)

    #updating the last processed timestamp in dynmodb
    #df.write.mode("overwrite").parquet(s3_target_path)

  

for table in table_name:
        if table["source_type"]=="my_sql":
            source_table_name=table["source_table"]
            target_table=table["target_table"]

            s3_bronze_path=f"{bronze_path}/{target_table}/{timestamp}"
            load_mysql_data=ingest_data_from_mysql_database(source_table_name,s3_bronze_path) 

            print(load_mysql_data)
        elif table["source_type"]=="s3":
            source_table_name=table["source_table"]
            target_table=table["target_table"]

            s3_path=f"{base_path}/{table['source_table']}"
            s3_bronze_path=f"{bronze_path}/{target_table}/{timestamp}"


            load_s3_data=ingest_data_from_s3(source_table_name,s3_path,s3_bronze_path)
            print(load_s3_data)
    

        
       




INFO:root:starting etl ingestion


None
None


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/share/aws/aws-java-sdk-v2/aws-sdk-java-bundle-2.29.52.jar!/software/amazon/awssdk/thirdparty/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/share/aws/glue-pds/jars/bundle-2.24.6.jar!/software/amazon/awssdk/thirdparty/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


{'Item': {'last_run_status': 'intial', 'last_run_timestamp': '1970-01-01 00:00:00', 'Source_name': 'orders', 'pipeline_table': 'orders', 'record_processed': Decimal('0'), 'Source_type': 's3', 'last_successful_watermark': '1970-01-01 00:00:00'}, 'ResponseMetadata': {'RequestId': '6J1QSGLPSHLKT0IF5JDAP2BNI7VV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sat, 22 Aug 2026 09:25:07 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '263', 'connection': 'keep-alive', 'x-amzn-requestid': '6J1QSGLPSHLKT0IF5JDAP2BNI7VV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '28339119'}, 'RetryAttempts': 0}}
1970-01-01 00:00:00


None


...

INFO:__main__:no incremental data found                                         


None


{'Item': {'Source_name': 'orders', 'Source_type': 's3', 'last_run_status': 'SUCCESS', 'last_run_timestamp': '2026-08-22 09:25:24', 'last_successful_watermark': '2026-07-08 09:10:00', 'pipeline_table': 'orders', 'record_processed': Decimal('0'), 'records_processed': Decimal('6')}, 'ResponseMetadata': {'RequestId': 'M2E6SMDPEGFR9FHQ3T78DRE66FVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sat, 22 Aug 2026 09:25:39 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '294', 'connection': 'keep-alive', 'x-amzn-requestid': 'M2E6SMDPEGFR9FHQ3T78DRE66FVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2971370719'}, 'RetryAttempts': 0}}
2026-07-08 09:10:00


INFO:__main__:no incremental data found                                         


None
